<a href="https://colab.research.google.com/github/sreejarepo/LLM_project1/blob/main/LLM_project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hello, GitHub! My project is running successfully.")

Hello, GitHub! My project is running successfully.


In [6]:
import os
import re
import hashlib
import gdown
from pypdf import PdfReader
from langdetect import detect, DetectorFactory

# Set seed for reproducible language detection
DetectorFactory.seed = 42

# =====================================================================
# STEP 1 — DOMAIN DATA COLLECTION & CLEANING
# =====================================================================

FOLDER_ID = '14Q_VaCEmqmQNmLNJlBOBzc6iHeylVoWU'
FOLDER_URL = f'https://google.com{FOLDER_ID}'

raw_pdf_dir = 'raw_pdfs/'
os.makedirs(raw_pdf_dir, exist_ok=True)

print("--- Step 1.1: Downloading compliance folders from Google Drive ---")
try:
    # Use remaining_ok=True and ensure gdown pulls all nested subfolders
    gdown.download_folder(url=FOLDER_URL, output=raw_pdf_dir, quiet=False, remaining_ok=True)
    print("Download operation completed.")
except Exception as e:
    print(f"Download execution problem encountered: {e}")

# Debugging step: Print exactly what is inside raw_pdfs to ensure files exist
print("\n--- Verifying downloaded file paths ---")
all_downloaded_items = []
for root, dirs, files in os.walk(raw_pdf_dir):
    for file in files:
        full_path = os.path.join(root, file)
        all_downloaded_items.append(full_path)
print(f"Found {len(all_downloaded_items)} total files in the download folder path.")

if len(all_downloaded_items) == 0:
    print("❌ WARNING: No files were downloaded! Please ensure your Google Drive folder sharing permissions are set to 'Anyone with the link can view'.")

# 2. Setup Data Structures & Pipeline Thresholds
output_corpus_dir = 'domain_corpus/'
os.makedirs(output_corpus_dir, exist_ok=True)

# Threshold justification: 200 characters prevents empty placeholders, signature sheets, or covers.
MIN_CHAR_LENGTH = 200
seen_page_hashes = set()

stats = {
    "raw_pdfs": 0,
    "total_pages_read": 0,
    "filtered_short": 0,
    "filtered_duplicate": 0,
    "filtered_non_english": 0,
    "extracted_txt_files": 0
}

# Recursively locate every .pdf across all subfolders (handles any nesting levels)
pdf_paths = []
for root, dirs, files in os.walk(raw_pdf_dir):
    for file in files:
        if file.endswith('.pdf'):
            pdf_paths.append(os.path.join(root, file))

stats["raw_pdfs"] = len(pdf_paths)
print(f"\n--- Step 1.2: Processing {len(pdf_paths)} collected PDF files recursively ---")

for pdf_path in pdf_paths:
    # Safely generate a flat text file filename using alphanumeric keys
    base_name = os.path.splitext(os.path.basename(pdf_path))[0]
    clean_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', base_name)
    txt_filename = f"{clean_name}.txt"
    txt_path = os.path.join(output_corpus_dir, txt_filename)

    cleaned_text_content = []

    try:
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages):
            stats["total_pages_read"] += 1
            text = page.extract_text() or ""
            text_striped = text.strip()

            # Filter 1: Length Threshold Filter
            if len(text_striped) < MIN_CHAR_LENGTH:
                stats["filtered_short"] += 1
                continue

            # Filter 2: Near-Duplicate/Identical Text Filter (MD5 Hash Matching)
            # Normalizing spaces and case ensures robust similarity matching
            normalized_text = re.sub(r'\s+', ' ', text_striped).lower()
            page_hash = hashlib.md5(normalized_text.encode('utf-8')).hexdigest()
            if page_hash in seen_page_hashes:
                stats["filtered_duplicate"] += 1
                continue
            seen_page_hashes.add(page_hash)

            # Filter 3: Language Check Isolation Filter (English Only)
            try:
                if detect(text_striped) != 'en':
                    stats["filtered_non_english"] += 1
                    continue
            except:
                stats["filtered_non_english"] += 1
                continue

            cleaned_text_content.append(text_striped)

        # Write matching target texts out to disk
        if cleaned_text_content:
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write("\n\n--- PAGE BREAK ---\n\n".join(cleaned_text_content))
            stats["extracted_txt_files"] += 1

    except Exception as e:
        print(f"Skipping corrupt or protected document {os.path.basename(pdf_path)}: {e}")

# Report Corpus Statistics
print("\n=== PART A: PIPELINE CORPUS STATISTICS ===")
print(f"Initial PDF files identified: {stats['raw_pdfs']}")
print(f"Total processed pages: {stats['total_pages_read']}")
print(f"Pages dropped via Length Filter (< {MIN_CHAR_LENGTH} chars): {stats['filtered_short']}")
print(f"Pages dropped via Content Deduplication Hashing: {stats['filtered_duplicate']}")
print(f"Pages dropped via Non-English Language Detection: {stats['filtered_non_english']}")
print(f"Final clean text logs exported to '{output_corpus_dir}': {stats['extracted_txt_files']} files")



--- Step 1.1: Downloading compliance folders from Google Drive ---


Retrieving folder contents


Download execution problem encountered: HTTPSConnectionPool(host='google.com14q_vacemqmqnmlnjlbobzc6iheylvowu', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7a537a011d10>: Failed to resolve 'google.com14q_vacemqmqnmlnjlbobzc6iheylvowu' ([Errno -2] Name or service not known)"))

--- Verifying downloaded file paths ---
Found 0 total files in the download folder path.
❌ WARNING: No files were downloaded! Please ensure your Google Drive folder sharing permissions are set to 'Anyone with the link can view'.

--- Step 1.2: Processing 0 collected PDF files recursively ---

=== PART A: PIPELINE CORPUS STATISTICS ===
Initial PDF files identified: 0
Total processed pages: 0
Pages dropped via Length Filter (< 200 chars): 0
Pages dropped via Content Deduplication Hashing: 0
Pages dropped via Non-English Language Detection: 0
Final clean text logs exported to 'domain_corpus/': 0 files
